# 官方成交量參考資料：正式批次（可分段）

把全期每個交易日的證交所與櫃買公開報表抓下來，存成版本化參考資料，再和缺棒帳本逐日對帳。

**怎麼跑**
- 每次開啟依序跑第 1–6 格。第 5 格會在 `MAX_MINUTES` 到時自動停在日期之間。
- 已完成的日期會跳過，所以隨時可以中斷，下次再跑第 1–6 格接著抓。
- 全部抓完（第 4 格顯示 `remaining: 0`）後跑第 7、8 格做全期對帳。

**資料處理**
- 證交所與櫃買同時抓，各自保持請求間隔（證交所 4 秒）。每天約 20 秒，全期約 4.5–5 小時。
- 每個請求的原始回應都壓縮保存，附 SHA-256、抓取時間與可得時間。
- 抓失敗會自動重試。連續 3 天失敗就自動停止（可能被限流），之後再跑即可。

**寫入範圍**
- 只寫到 `_tw-market-data-lake/reference/official_volume/v1/`。
- 不碰任何 lake 資料集、raw Parquet、manifest 或 published pointer。
- 不需要任何憑證，不呼叫富邦。

In [ ]:
# 1. 參數
REPO_OWNER, REPO_NAME, BRANCH = "hh4832", "tw-market-data-lake", "main"
DATA_ROOT = "/content/drive/MyDrive/00Quant_Research/data"
LEDGER_DIR = f"{DATA_ROOT}/_tw-market-data-lake/audits/gap_ledger_20261026_163358_3e8eed0"
DATASET_ROOT = f"{DATA_ROOT}/_tw-market-data-lake/reference/official_volume/v1"
MAX_MINUTES = 55          # 這次最多抓多久（分鐘）；到時停在兩個日期之間
NEWEST_FIRST = False      # True = 從最近的日期往回抓
PROJECT_DIR = f"/content/{REPO_NAME}"
OUT_DIR = "/content/tw_lake_volume_recon"

In [ ]:
# 2. 掛載 Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 3. clone / pull + 依賴
import subprocess, os
if os.path.exists(f"{PROJECT_DIR}/.git"):
    subprocess.run(["git", "-C", PROJECT_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git", PROJECT_DIR], check=True)
print("HEAD:", subprocess.check_output(["git", "-C", PROJECT_DIR, "rev-parse", "HEAD"], text=True).strip())
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml>=6", "tqdm>=4.66"], check=True)
def stream(args):
    env = dict(os.environ, PYTHONPATH=f"{PROJECT_DIR}/src", PYTHONUNBUFFERED="1")
    p = subprocess.Popen(args, cwd=PROJECT_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    return p.wait()

def ensure_drive():
    """Mount Drive safely. If an earlier cell unmounted Drive and then wrote under /content/drive,
    those files are only on this runtime's disk: move them aside (never delete) and copy any audit
    folders that are missing on the real Drive."""
    import os, shutil, time
    from google.colab import drive
    if os.path.ismount("/content/drive"):
        return
    stray = None
    if os.path.isdir("/content/drive") and os.listdir("/content/drive"):
        stray = f"/content/drive_local_{int(time.time())}"
        shutil.move("/content/drive", stray)
        print("moved local (unsynced) files to", stray)
    drive.mount("/content/drive")
    if stray:
        src_root = f"{stray}/MyDrive/00Quant_Research/data/_tw-market-data-lake/audits"
        dst_root = f"{DATA_ROOT}/_tw-market-data-lake/audits"
        for dirpath, dirnames, filenames in os.walk(src_root):
            rel = os.path.relpath(dirpath, src_root)
            for d in list(dirnames):
                target = os.path.join(dst_root, rel, d)
                if not os.path.exists(target):
                    shutil.copytree(os.path.join(dirpath, d), target)
                    print("copied missing", target)
                    dirnames.remove(d)

In [ ]:
# 4. 進度
print("exit", stream([sys.executable, "-m", "tw_lake.reference.official_volume_batch", "status",
                      "--ledger-dir", LEDGER_DIR, "--dataset-root", DATASET_ROOT]))

In [ ]:
# 5. 抓取（到 MAX_MINUTES 自動停止；可隨時中斷，已完成的日期不會重抓）
ensure_drive()
args = [sys.executable, "-m", "tw_lake.reference.official_volume_batch", "fetch", "--ledger-dir", LEDGER_DIR,
        "--dataset-root", DATASET_ROOT, "--max-minutes", str(MAX_MINUTES)]
if NEWEST_FIRST:
    args.append("--newest-first")
print("exit", stream(args))

In [ ]:
# 6. 確保上傳到 Drive（每次抓完都要跑）
from google.colab import drive
drive.flush_and_unmount()
print("flushed; 下次重跑第 1–6 格即可接著抓")

In [ ]:
# 7. 全期對帳（無網路；全部抓完後再跑）
import glob, json, os
import pandas as pd
ensure_drive()
print("exit", stream([sys.executable, "-m", "tw_lake.reference.official_volume_batch", "recon",
                      "--ledger-dir", LEDGER_DIR, "--dataset-root", DATASET_ROOT, "--out-dir", OUT_DIR]))
RECON_DIR = sorted(glob.glob(f"{OUT_DIR}/volume_recon_*"))[-1]
pd.set_option("display.width", 250)
bd = pd.read_csv(f"{RECON_DIR}/identity_by_date.csv")
print(bd.groupby(["exchange", "date_pattern"]).size().unstack(fill_value=0))
print(bd[bd["date_pattern"] == "formula_unverified"].to_string(index=False))
s = json.load(open(f"{RECON_DIR}/volume_recon_summary.json"))
print(pd.DataFrame(s["missing_days"]["by_class"]).to_string(index=False))
print({k: v for k, v in s["missing_days"].items() if k != "by_class"})

In [ ]:
# 8. 複製對帳結果到 Drive
import shutil
dest = f"{DATA_ROOT}/_tw-market-data-lake/audits/official_volume_batch_recon/{os.path.basename(RECON_DIR)}"
shutil.copytree(RECON_DIR, dest)
print("copied to", dest)
from google.colab import drive
drive.flush_and_unmount()